## Bronze: raw ingestion
Reads the three source tables from Postgres via JDBC and writes each as a Delta table, untouched, to preserve raw history.

In [0]:
%python
# Connection parameters
jdbc_url = "jdbc:postgresql://aws-0-eu-west-1.pooler.supabase.com:5432/postgres"
properties = {
    "user": "interview_candidate.qpofesblybgqvdknsrlp",
    "password": "cs26_pehF7EBQDStnowI9zv1x",
    "driver": "org.postgresql.Driver",
    "ssl": "true",
    "sslmode": "require"
}

# Ingest tables
tables = ["bot_logs", "price_log", "agent_decisions"]

for table in tables:
    df = spark.read.jdbc(url=jdbc_url, table=f"public.{table}", properties=properties)
    
    # Save as managed Bronze Delta table
    df.write.format("delta") \
        .mode("overwrite") \
        .saveAsTable(f"bronze_{table}")

Sanity check — confirm the bronze ingest landed correctly before building on top of it.

In [0]:
select *
from workspace.default.bronze_bot_logs

## Silver: signal spine
One row per signal, typed and flagged rather than filtered — `is_btc`, `is_v2_era`, and `is_outage_window` encode the Part 1 findings (symbol scope, bot-roster rollout, and the May–Sept outage) so downstream queries can filter explicitly instead of relying on hidden assumptions. `decision_is_informative` flags the three always-Hold bots whose decisions carry no signal.

In [0]:
create or replace table silver_signals as
select
    id                              as signal_id,
    timestamp                       as signaled_at,
    bot_name,
    decision,                       -- 'Buy' | 'Hold' | 'Sell', kept as-is
    current_price::numeric,
    scoped_symbol                   as symbol,
    scoped_metric,                  -- kept as JSONB, untouched, for silver_metrics to explode

    -- flags derived from Part 1 findings, not judgment calls buried in WHERE clauses
    (scoped_symbol = 'BTC/USDT')                        as is_btc,
    (timestamp >= '2026-04-03')                          as is_v2_era,   -- ensemble rollout date
    (timestamp between '2026-05-08' and '2026-09-03')    as is_outage_window,

    -- which bots' "decision" column is actually informative (from Part 1: always-Hold bots)
    (bot_name not in (
        'AnomalyDetectionBot', 'BayesianCompositeBot', 'StatsForecastBot'
    ))                                                    as decision_is_informative

from bronze_bot_logs;

## Silver: long-form metrics
Explodes `scoped_metric` into one row per (signal, metric key) instead of flattening into columns, so new bots or new metric keys never require a schema change. Non-numeric values (e.g. `regime: "bullish"`) are kept in `raw_value` with `numeric_value` as NULL.

In [0]:
create or replace table silver_metrics as
select
    s.signal_id,
    s.bot_name,
    s.signaled_at,
    kv.key                                  as metric_name,
    kv.value                                as raw_value,        -- string form of the value
    try_cast(kv.value as double)            as numeric_value     -- NULL if it's not numeric, e.g. 'regime': 'bull'
from silver_signals s
lateral view explode(
    from_json(s.scoped_metric, 'map<string,string>')
) kv as key, value;

QA check — spot-check the exploded metrics look correct.

In [0]:
select * from workspace.default.silver_metrics

QA check — confirm no rows failed to parse as JSON (would indicate a nested/array value `from_json` can't handle as a flat map).

In [0]:
select bot_name, scoped_metric
from silver_signals
where from_json(scoped_metric, 'map<string,string>') is null
limit 20;

## Gold: shared price timeline
Combines `current_price` from every bot signal (dense, continuous from Feb 2025) with `price_log` (sparse, April 2026 onward) into one deduplicated price series. Used as the single source of truth for label construction below.

In [0]:
create or replace table gold_price_timeline as
select distinct
    signaled_at as price_time,
    current_price as price,
    'bot_logs' as source
from silver_signals
where is_btc

union all

select distinct
    timestamp as price_time,
    price,
    concat('price_log:', source) as source
from bronze_price_log
where symbol = 'BTC/USDT'
order by price_time;

QA check — inspect the combined timeline post-rollout, where both sources overlap.

In [0]:
select *
from workspace.default.gold_price_timeline
where price_time > '2026-04-04'

## Gold: MA/EMA features and label
Three tables, built in order:
- `gold_ma_ema_base` — one row per BasicMovingAverageBot/ExponentialMovingAverageBot signal, with the engineered `pct_diff_from_ma` feature. Scoped to BTC only, before the 2026-05-09 cutoff.
- `gold_bot_agreement` — ensemble agreement, computed only from 2026-04-03 onward (undefined before the rollout), excluding the three always-Hold bots so they don't mechanically skew the percentage.
- `gold_ma_ema_features_labels` — joins features, agreement, and the 24h-forward label via an as-of price lookup capped at a 3-hour tolerance, so signals near the April data gap are dropped rather than mislabeled.

In [0]:
-- ============================================================
-- GOLD: MA / EMA pooled features + label
-- Bots: BasicMovingAverageBot, ExponentialMovingAverageBot
-- Label: 24h-forward price direction
-- Cutoff: signals before 2026-05-09 (matches brief; stops before the outage)
-- ============================================================

create or replace table gold_ma_ema_base as
select
    s.signal_id,
    s.signaled_at,
    s.bot_name,
    case when s.bot_name = 'BasicMovingAverageBot' then 'basic' else 'exponential' end as bot_type,
    s.current_price,
    m.numeric_value                                            as moving_average,
    (s.current_price - m.numeric_value) / m.numeric_value       as pct_diff_from_ma,
    s.decision                                                  as bot_decision,   -- "follow-the-bot" baseline
    (s.signaled_at >= timestamp'2026-04-03')                    as is_v2_era
from silver_signals s
join silver_metrics m
  on m.signal_id = s.signal_id and m.metric_name = 'moving_average'
where s.bot_name in ('BasicMovingAverageBot', 'ExponentialMovingAverageBot')
  and s.is_btc
  and s.signaled_at < timestamp'2026-05-09';

-- Bot agreement: only computed post-rollout, since pre-04-03 only these 4 bots exist
-- (asking "do other bots agree" pre-rollout is circular/meaningless)
create or replace table gold_bot_agreement as
select
    base.signal_id,
    count(distinct ob.bot_name)                                as active_bot_count,
    sum(case when ob.decision = 'Buy'  then 1 else 0 end)
        / nullif(count(distinct ob.bot_name), 0)                as pct_buy_agreement
from gold_ma_ema_base base
join lateral (
    select b.bot_name, b.decision,
           row_number() over (partition by b.bot_name order by b.signaled_at desc) as rn
    from silver_signals b
    where b.signaled_at <= base.signaled_at
      and b.is_btc
      and b.bot_name not in (base.bot_name, 'AnomalyDetectionBot', 'BayesianCompositeBot', 'StatsForecastBot')
) ob
where ob.rn = 1
  and base.is_v2_era
group by base.signal_id;

-- Final feature + label table
create or replace table gold_ma_ema_features_labels as
select
    base.*,
    agr.active_bot_count,
    agr.pct_buy_agreement,
    fp.price       as future_price,
    fp.price_time  as future_price_time,
    case when fp.price > base.current_price then 1 else 0 end as label_up
from gold_ma_ema_base base
left join gold_bot_agreement agr on agr.signal_id = base.signal_id
join lateral (
    select price, price_time
    from gold_price_timeline t
    where t.price_time >= base.signaled_at + interval 24 hours
      and t.price_time <  base.signaled_at + interval 27 hours   -- ← the fix: 3-hour tolerance
    order by t.price_time asc
    limit 1
) fp
order by base.signaled_at;

## Model training and evaluation
Trains separate logistic regression models for the pre- and post-rollout eras, each against three baselines (always-up, majority class, follow-the-bot). Splits are chronological, not random, since overlapping 24h label windows would otherwise leak test-period information into training.

In [0]:
%python
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# Assumes gold_ma_ema_features_labels has been materialized and pulled locally, e.g.:
df = spark.table("gold_ma_ema_features_labels").toPandas()
df = df.sort_values("signaled_at").reset_index(drop=True)
df["bot_type_encoded"] = (df["bot_type"] == "exponential").astype(int)
df["bot_decision_as_pred"] = df["bot_decision"].map({"Buy": 1, "Sell": 0})  # MA/EMA never emit Hold

def chronological_split(data, test_frac=0.2):
    """Split by TIME, not row count or shuffle — respects the overlapping-label issue."""
    cutoff = data["signaled_at"].quantile(1 - test_frac)
    train = data[data["signaled_at"] < cutoff]
    test = data[data["signaled_at"] >= cutoff]
    return train, test

def evaluate(y_true, y_pred, y_proba=None, label="model"):
    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
    }
    if y_proba is not None and y_true.nunique() > 1:
        metrics["roc_auc"] = roc_auc_score(y_true, y_proba)
    print(f"  [{label:20s}] " + "  ".join(f"{k}={v:.3f}" for k, v in metrics.items()))
    return metrics

def run_era(data, feature_cols, era_name, balanced=False):
    print(f"\n=== {era_name} (n={len(data)}) ===")
    train, test = chronological_split(data)
    y_train, y_test = train["label_up"], test["label_up"]

    # baselines unchanged...
    always_up = np.ones(len(test))
    evaluate(y_test, always_up, label="baseline: always up")
    majority = int(y_train.mode()[0])
    evaluate(y_test, np.full(len(test), majority), label="baseline: majority class")
    bot_pred = test["bot_decision_as_pred"].fillna(majority).astype(int)
    evaluate(y_test, bot_pred, label="baseline: follow bot")

    X_train, X_test = train[feature_cols], test[feature_cols]
    scaler = StandardScaler().fit(X_train)
    X_train_s, X_test_s = scaler.transform(X_train), scaler.transform(X_test)

    model = LogisticRegression(max_iter=1000, class_weight="balanced" if balanced else None)
    model.fit(X_train_s, y_train)

    y_pred = model.predict(X_test_s)
    y_proba = model.predict_proba(X_test_s)[:, 1]
    label = "logistic regression (balanced)" if balanced else "logistic regression"
    evaluate(y_test, y_pred, y_proba, label=label)

    return model, scaler
    
# --- Pre-April-3: only pct_diff_from_ma and bot_type available (no agreement feature) ---
pre_df = df[~df["is_v2_era"]].copy()
pre_model, pre_scaler = run_era(
    pre_df,
    feature_cols=["pct_diff_from_ma", "bot_type_encoded"],
    era_name="PRE 2026-04-03 (legacy, 4-bot universe)"
)

# --- Post-April-3: full feature set including bot agreement ---
post_df = df[df["is_v2_era"]].copy()
post_model, post_scaler = run_era(
    post_df,
    feature_cols=["pct_diff_from_ma", "bot_type_encoded", "pct_buy_agreement", "active_bot_count"],
    era_name="POST 2026-04-03 (full ensemble)"
)

## Final model: leakage check, refit, and serialization
Re-verifies the train/test time boundary, refits the pre-rollout model on the full training split (chosen for deployment over the post-rollout model, which collapsed during training), and serializes the model and scaler for the Streamlit app.

In [0]:
%python
import joblib
import json
import pandas as pd
import os

os.makedirs("models", exist_ok=True)
os.makedirs("reports", exist_ok=True)

# ── 1. Leakage sanity check ──────────────────────────────────────────
assert pre_df["signaled_at"].max() < pd.Timestamp("2026-05-09"), "cutoff violated"
train_final, test_final = chronological_split(pre_df)
assert train_final["signaled_at"].max() < test_final["signaled_at"].min(), \
    "train/test overlap in time — leakage risk"
print(f"Leakage check passed: train ends {train_final.signaled_at.max()}, "
      f"test starts {test_final.signaled_at.min()}")

# ── 2. Refit final shipped model ──────────────────────────────────────
feature_cols = ["pct_diff_from_ma", "bot_type_encoded"]
X_train, X_test = train_final[feature_cols], test_final[feature_cols]
y_train, y_test = train_final["label_up"], test_final["label_up"]

final_scaler = StandardScaler().fit(X_train)
final_model = LogisticRegression(max_iter=1000).fit(final_scaler.transform(X_train), y_train)

# ── 3. Serialize for Part 3 ──────────────────────────────────────────
joblib.dump(final_model, "models/ma_ema_model.joblib")
joblib.dump(final_scaler, "models/ma_ema_scaler.joblib")

# ── 4. Summary table for the README ──────────────────────────────────
def summarize(data, model, scaler, feature_cols, label):
    train, test = chronological_split(data)
    X_test_s = scaler.transform(test[feature_cols])
    y_test = test["label_up"]
    y_pred = model.predict(X_test_s)
    y_proba = model.predict_proba(X_test_s)[:, 1]
    bot_pred = test["bot_decision_as_pred"].fillna(int(train["label_up"].mode()[0])).astype(int)

    return {
        "era": label,
        "n_test": len(test),
        "up_label_rate": round(y_test.mean(), 3),
        "baseline_majority_acc": round(max(y_test.mean(), 1 - y_test.mean()), 3),
        "baseline_follow_bot_acc": round(accuracy_score(y_test, bot_pred), 3),
        "model_acc": round(accuracy_score(y_test, y_pred), 3),
        "model_roc_auc": round(roc_auc_score(y_test, y_proba), 3) if y_test.nunique() > 1 else None,
    }

summary = pd.DataFrame([
    summarize(pre_df, final_model, final_scaler, feature_cols, "pre-2026-04-03"),
])
summary.to_csv("reports/model_summary.csv", index=False)
print(summary.to_markdown(index=False))

## Export for Streamlit
Small CSV export of signal history for the app's Explore tab.

In [0]:
%python
history_cols = ["signaled_at", "current_price", "moving_average", "pct_diff_from_ma"]

df = spark.table("workspace.default.gold_ma_ema_features_labels") \
    .select(*history_cols) \
    .toPandas()

df.to_csv("ma_ema_history.csv", index=False)